<a href="https://colab.research.google.com/github/l22140603/Analisis-y-visualizacion-de-datos/blob/main/HMGG_Parte_2_Ejercicio_de_Regresi%C3%B3n_en_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install.packages("ISLR2")
install.packages("ggplot2")
install.packages("readr")

# Cargar librerías necesarias
#library(ISLR2)
library(ggplot2)
library(readr)
# Cargar datos de publicidad
#data(Advertising) # Variables: TV, Radio, Newspaper, Sales
#data(Credit)
Advertising<- read.csv("Advertising.csv")



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘vroom’


Warning message in file(file, "rt"):
“cannot open file 'Advertising.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [ ]:
# Ajustar modelo de regresión lineal simple
# Predecir ventas en función del presupuesto de TV
modelo_simple <- lm(Sales ~ TV, data = Advertising)
# Ver resumen completo del modelo
summary(modelo_simple)

In [ ]:
# Visualizar la recta de regresión
ggplot(Advertising, aes(x = TV, y = Sales)) + geom_point(color = '#2E75B6', alpha = 0.6, size = 2.5) + geom_smooth(method = 'lm', color = '#E74C3C', se = TRUE) + labs(title = 'Ventas vs. Inversión en TV', x = 'Presupuesto TV (miles USD)', y = 'Ventas (miles de unidades)') + theme_minimal()

In [ ]:
# Modelo de regresión múltiple
modelo_mult <- lm(Sales ~ TV + Radio + Newspaper, Advertising)
summary(modelo_mult)
# Resultado esperado (aproximado):
# Coefficients: Estimate Std. Error t value Pr(&gt;|t|)
# (Intercept) 2.9389 0.3119 9.422 &lt;2e-16 ***
# TV 0.0458 0.0014 32.809 &lt;2e-16 ***
# radio 0.1885 0.0086 21.893 &lt;2e-16 ***
# newspaper -0.0010 0.0059 -0.177 0.860
# ---
# Multiple R-squared: 0.8972, Adjusted R-squared: 0.8956
# Predicción para nuevos datos
nuevos_datos <- data.frame(TV = 150, Radio = 25, Newspaper = 30)
predict(modelo_mult, newdata = nuevos_datos, interval = 'prediction', level = 0.95)

In [ ]:
# Ejemplo con variable categórica: ShelveLoc (Good/Bad/Medium)
Carseats<- read.csv("Carseats.csv")


In [ ]:
modelo_carseat <- lm(Sales ~ Price + ShelveLoc + Age + Income, data = Carseats)
summary(modelo_carseat)


In [ ]:
# R crea automáticamente:
# ShelveLocGood → 1 si ubicación es &#39;Good&#39;, 0 si no
# ShelveLocMedium → 1 si ubicación es &#39;Medium&#39;, 0 si no
# (Bad es la categoría de referencia)
# Verificar qué contraste usa R
contrasts(as.factor(Carseats$ShelveLoc))

In [ ]:
# Interacción entre TV y radio
# El efecto de TV puede amplificarse cuando radio también es alto
modelo_interact <- lm(Sales ~ TV * Radio, data = Advertising)
summary(modelo_interact)
# Equivalente a:
# lm(sales ~ TV + radio + TV:radio, data = Advertising)
# Modelo con transformación logarítmica y cuadrática
modelo_poly <- lm(Sales ~ TV + I(TV^2) + Radio, data = Advertising)
summary(modelo_poly)

In [ ]:
# Gráficos de diagnóstico completos
par(mfrow = c(2, 2)) # Layout 2x2
plot(modelo_mult) # 4 gráficos automáticos
# Gráfico 1 - Residuals vs Fitted: detecta no linealidad
# Gráfico 2 - Q-Q Plot: evalúa normalidad de residuos
# Gráfico 3 - Scale-Location: evalúa homocedasticidad
# Gráfico 4 - Residuals vs Leverage: detecta puntos influyentes


In [ ]:
# ── Prueba formal de homocedasticidad ──
install.packages("lmtest")
library(lmtest)
bptest(modelo_mult) # Breusch-Pagan: H0 = homocedasticidad
# ── Multicolinealidad ──
install.packages("car")
library(car)
vif(modelo_mult) # VIF &lt; 5 = aceptable, &gt;10 = problema severo
# ── Observaciones influyentes ──
influenceIndexPlot(modelo_mult, vars = c('Cook', 'Studentized'))
# Distancia de Cook &gt; 1 sugiere observación muy influyente

# Parte 2

In [ ]:
install.packages("caret")

In [ ]:
install.packages("Metrics")

In [ ]:
install.packages("readr")

In [ ]:
install.packages("ISLR2")

In [ ]:
library(ISLR2)
data(Advertising)

In [ ]:
library(caret)
library(Metrics)
library(readr)
library(ISLR2)
# ── Método 1: Train/Test Split (70/30) ──
set.seed(123) # Reproducibilidad
Advertising<-read.csv("Advertising.csv")
n <- nrow(Advertising)
train_idx <- sample(1:n, size = 0.7 * n)
train_data <- Advertising[train_idx, ]
test_data <- Advertising[-train_idx, ]
modelo_train <- lm(Sales ~ TV + Radio + Newspaper, data = train_data)
pred_test <- predict(modelo_train, newdata = test_data)


In [ ]:
# Métricas en test set
rmse_test <- rmse(test_data$Sales, pred_test)
mae_test <- mae(test_data$Sales, pred_test)
cat('RMSE test:', round(rmse_test, 3), '\n')
cat('MAE test:', round(mae_test, 3), '\n')


In [ ]:
head(train_data)

In [ ]:
# ── Método 2: Validación Cruzada 10-Fold ──
control <- trainControl(method = 'cv', number = 10)
modelo_cv <- train(Sales ~ TV + Radio + Newspaper, method = 'lm', trControl = control, data=Advertising)
print(modelo_cv) # Muestra RMSE promedio de los 10 folds

In [ ]:
install.packages("MASS")

In [ ]:
install.packages("leaps")

In [ ]:
library(MASS) # Para stepAIC
library(leaps) # Para regsubsets (mejor subconjunto)
# ── Selección Stepwise (AIC) ──
modelo_full <- lm(Sales ~ ., data = Advertising)
modelo_step <- stepAIC(modelo_full, direction = 'both', trace = FALSE)
summary(modelo_step)

In [ ]:
# ── Mejor Subconjunto (Exhaustivo) ──
reg_subset <- regsubsets(Sales ~ ., data = Advertising, nvmax = 4)
reg_summary <- summary(reg_subset)


In [ ]:
# Identificar mejor modelo según BIC
mejor_bic <- which.min(reg_summary$bic)
coef(reg_subset, mejor_bic)


In [ ]:
# ── Comparar modelos con ANOVA ──
modelo_reducido <- lm(Sales ~ TV + Radio, data = Advertising)
modelo_completo <- lm(Sales ~ TV + Radio + Newspaper, data = Advertising)
anova(modelo_reducido, modelo_completo)
# Si p < 0.05 → el modelo reducido es preferible

In [ ]:
# Ejemplo: modelo de demanda con elasticidad precio
# log(Ventas) = β₀ + β₁·log(Precio) + β₂·Promo + β₃·Estacion + ε
# Generar datos de ejemplo
set.seed(42)
n <- 120 # 10 años de datos mensuales
precio <- runif(n, 10, 50)
promo <- rbinom(n, 1, 0.3) # 30% de meses con promoción
estacion <- factor(rep(1:4, length.out = n)) # Trimestres
ventas <- 5000 * precio^(-1.3) * exp(0.4*promo) * rnorm(n, 1, 0.05)
df_ventas <- data.frame(ventas, precio, promo, estacion)
# Modelo log-log (elasticidad constante)
modelo_demanda <- lm(log(ventas) ~ log(precio) + promo + estacion,
data = df_ventas)
summary(modelo_demanda)
# Interpretación: β₁ = elasticidad precio
# Si β₁ = -1.3 → un aumento del 1% en precio reduce ventas 1.3%

In [ ]:
# Ejemplo: modelo de tiempo de ciclo en manufactura
# Predecir tiempo de ciclo (min) desde variables de proceso
set.seed(7)
n <- 200
temperatura <- runif(n, 180, 240) # °C
presion <- runif(n, 50, 120) # PSI
velocidad <- runif(n, 10, 80) # RPM
operador_exp <- sample(1:5, n, replace = TRUE) # años experiencia
tiempo_ciclo <- 45 + 0.1*temperatura - 0.05*presion + 0.2*velocidad - 1.5*operador_exp + rnorm(n, 0, 3)
df_proceso <- data.frame(tiempo_ciclo, temperatura, presion, velocidad, operador_exp)
modelo_proceso <- lm(tiempo_ciclo ~ ., data = df_proceso)
summary(modelo_proceso)
# Predicción para configuración óptima
config_optima <- data.frame(temperatura = 200, presion = 100,velocidad = 50, operador_exp = 4)
predict(modelo_proceso, newdata = config_optima, interval = 'prediction', level = 0.95)

In [ ]:
install.packages("tidyverse")

In [ ]:
install.packages("broom")

In [ ]:
library(tidyverse)
library(broom) # Para convertir outputs de lm() a tibbles
# Ajustar y extraer resultados en formato tidy
modelo_tidy <- lm(Sales ~ TV + Radio, data = Advertising)


In [ ]:
# Tabla de coeficientes limpia
tidy(modelo_tidy, conf.int = TRUE) %>%
  mutate(across(where(is.numeric), ~round(., 4)))


In [ ]:
# Métricas de ajuste del modelo
glance(modelo_tidy)


In [ ]:
# Valores ajustados y residuos
augment(modelo_tidy) %>%
  select(Sales, .fitted, .resid, .hat, .cooksd) %>%
  head(10)


In [ ]:
# Gráfico de coeficientes con intervalos de confianza
tidy(modelo_tidy, conf.int = TRUE) %>%
  filter(term != '(Intercept)') %>%
  ggplot(aes(x = estimate, y = term)) + geom_point(color = '#2E75B6', size = 3) + geom_errorbarh(aes(xmin = conf.low, xmax = conf.high), height = 0.2) + geom_vline(xintercept = 0, linetype = 'dashed', color = 'red') +
  labs(title = 'Coeficientes con IC 95%', x = 'Estimación', y = '') + theme_minimal()